In [1]:
import duckdb
import os

In [2]:
DATA_DIR = "data"
OPT_FILE = os.path.join(DATA_DIR, "option_prices.csv.gz")
OUT_DIR  = os.path.join(DATA_DIR, "processed")
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
con = duckdb.connect(database=os.path.join(OUT_DIR, "wrk.duckdb"))

In [4]:
files = {
    "option_prices": os.path.join(DATA_DIR, "option_prices.csv.gz"),
    "volatility_surface": os.path.join(DATA_DIR, "volatility_surface.csv.gz"),
    "historical_volatility": os.path.join(DATA_DIR, "historical_volatility.csv.gz"),
    "option_volume": os.path.join(DATA_DIR, "option_volume.csv.gz"),
    "security_prices": os.path.join(DATA_DIR, "security_prices.csv.gz"),
    "security_name_history": os.path.join(DATA_DIR, "security_name_history.csv.gz"),
    "crsp_daily_stock": os.path.join(DATA_DIR, "crsp_daily_stock.csv.gz"),
    "short_interest": os.path.join(DATA_DIR, "short_interest.csv.gz"),
    "compustat_fundq": os.path.join(DATA_DIR, "compustat_fundamentals_quarterly.csv.gz"),
    "ccm_link_table": os.path.join(DATA_DIR, "ccm_link_table.csv.gz"),
}

In [5]:
con.execute(f"""
CREATE OR REPLACE TABLE option_prices_small AS
SELECT
    secid,
    CAST(date AS DATE) AS date,
    CAST(exdate AS DATE) AS exdate,
    cp_flag,
    strike_price,
    best_bid,
    best_offer,
    (best_bid + best_offer) / 2.0 AS mid,
    impl_volatility,
    delta,
    gamma,
    vega,
    theta,
    volume,
    open_interest,
    optionid,
    ticker,
    cusip,
    exercise_style,
    (CAST(exdate AS DATE) - CAST(date AS DATE)) AS dte
FROM read_csv_auto('{files["option_prices"]}', compression='gzip')
WHERE best_bid IS NOT NULL
  AND best_offer IS NOT NULL
  AND strike_price IS NOT NULL
  AND open_interest IS NOT NULL
  AND open_interest > 0
  AND delta IS NOT NULL
  AND ABS(delta) BETWEEN 0.20 AND 0.80
  AND (CAST(exdate AS DATE) - CAST(date AS DATE)) BETWEEN 7 AND 120
""")

con.execute(f"""
COPY option_prices_small TO '{os.path.join(OUT_DIR, "option_prices_small.parquet")}'
(FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
con.execute(f"""
CREATE OR REPLACE TABLE volatility_surface_small AS
SELECT
    secid,
    CAST(date AS DATE) AS date,
    days,
    delta,
    cp_flag,
    impl_volatility,
    impl_strike,
    impl_premium,
    ticker
FROM read_csv_auto('{files["volatility_surface"]}', compression='gzip')
WHERE days IN (30, 60, 91)
""")

con.execute(f"""
COPY volatility_surface_small TO '{os.path.join(OUT_DIR, "volatility_surface_small.parquet")}'
(FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
con.execute(f"""
CREATE OR REPLACE TABLE historical_volatility_small AS
SELECT
    secid,
    CAST(date AS DATE) AS date,
    days,
    volatility,
    ticker
FROM read_csv_auto('{files["historical_volatility"]}', compression='gzip')
WHERE days IN (10, 20, 30, 60, 90)
""")

con.execute(f"""
COPY historical_volatility_small TO '{os.path.join(OUT_DIR, "historical_volatility_small.parquet")}'
(FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
con.execute(f"""
CREATE OR REPLACE TABLE option_volume_small AS
SELECT
    secid,
    CAST(date AS DATE) AS date,
    cp_flag,
    volume,
    open_interest,
    ticker
FROM read_csv_auto('{files["option_volume"]}', compression='gzip')
""")

con.execute(f"""
COPY option_volume_small TO '{os.path.join(OUT_DIR, "option_volume_small.parquet")}'
(FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
con.execute(f"""
CREATE OR REPLACE TABLE security_prices_small AS
SELECT
    secid,
    CAST(date AS DATE) AS date,
    ticker,
    cusip,
    open,
    high,
    low,
    close,
    volume,
    return,
    shrout
FROM read_csv_auto('{files["security_prices"]}', compression='gzip')
""")

con.execute(f"""
COPY security_prices_small TO '{os.path.join(OUT_DIR, "security_prices_small.parquet")}'
(FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
con.execute(f"""
CREATE OR REPLACE TABLE security_name_history_small AS
SELECT
    secid,
    ticker,
    cusip,
    CAST(effect_date AS DATE) AS effect_date
FROM read_csv_auto('{files["security_name_history"]}', compression='gzip')
""")

con.execute(f"""
COPY security_name_history_small TO '{os.path.join(OUT_DIR, "security_name_history_small.parquet")}'
(FORMAT PARQUET)
""")

In [12]:
con.execute(f"""
CREATE OR REPLACE TABLE crsp_daily_stock_small AS
SELECT
    CAST(PERMNO AS BIGINT) AS permno,
    CAST(date AS DATE) AS date,
    TICKER,
    NCUSIP,
    COMNAM,
    CAST(SHRCD AS INTEGER) AS shrcd,
    CAST(EXCHCD AS INTEGER) AS exchcd,
    CAST(SICCD AS INTEGER) AS siccd,
    TRY_CAST(PRC AS DOUBLE) AS prc,
    RET AS ret_raw,
    TRY_CAST(RET AS DOUBLE) AS ret,
    TRY_CAST(VOL AS DOUBLE) AS vol,
    TRY_CAST(SHROUT AS DOUBLE) AS shrout,
    TRY_CAST(OPENPRC AS DOUBLE) AS openprc
FROM read_csv_auto(
    '{files["crsp_daily_stock"]}',
    compression='gzip',
    types={{
        'RET': 'VARCHAR',
        'PRC': 'VARCHAR',
        'VOL': 'VARCHAR',
        'SHROUT': 'VARCHAR',
        'OPENPRC': 'VARCHAR',
        'SHRCD': 'VARCHAR',
        'EXCHCD': 'VARCHAR',
        'SICCD': 'VARCHAR',
        'PERMNO': 'VARCHAR'
    }}
)
WHERE TRY_CAST(SHRCD AS INTEGER) IN (10, 11)
""")

con.execute(f"""
COPY crsp_daily_stock_small TO '{os.path.join(OUT_DIR, "crsp_daily_stock_small.parquet")}'
(FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
con.execute(f"""
CREATE OR REPLACE TABLE short_interest_small AS
SELECT
    gvkey,
    tic,
    CAST(datadate AS DATE) AS datadate,
    shortint,
    shortintadj
FROM read_csv_auto('{files["short_interest"]}', compression='gzip')
""")

con.execute(f"""
COPY short_interest_small TO '{os.path.join(OUT_DIR, "short_interest_small.parquet")}'
(FORMAT PARQUET)
""")

In [14]:
con.execute(f"""
CREATE OR REPLACE TABLE compustat_fundq_small AS
SELECT
    gvkey,
    tic,
    CAST(datadate AS DATE) AS datadate,
    CAST(rdq AS DATE) AS rdq,
    fyearq,
    fqtr,
    epspxq,
    prccq,
    mkvaltq
FROM read_csv_auto('{files["compustat_fundq"]}', compression='gzip')
WHERE rdq IS NOT NULL
""")

con.execute(f"""
COPY compustat_fundq_small TO '{os.path.join(OUT_DIR, "compustat_fundq_small.parquet")}'
(FORMAT PARQUET)
""")

In [5]:
con.execute(f"""
CREATE OR REPLACE TABLE ccm_link_table_small AS
SELECT
    gvkey,
    TRY_CAST(LPERMNO AS BIGINT) AS LPERMNO,
    LINKPRIM,
    LINKTYPE,
    TRY_CAST(LINKDT AS DATE) AS LINKDT,
    TRY_CAST(NULLIF(LINKENDDT, 'E') AS DATE) AS LINKENDDT
FROM read_csv_auto(
    '{files["ccm_link_table"]}',
    compression='gzip',
    types={{
        'LINKDT': 'VARCHAR',
        'LINKENDDT': 'VARCHAR'
    }}
)
WHERE LINKTYPE IN ('LU','LC')
  AND LINKPRIM = 'P'
""")

con.execute(f"""
COPY ccm_link_table_small TO '{os.path.join(OUT_DIR, "ccm_link_table_small.parquet")}'
(FORMAT PARQUET)
""")

In [6]:
for f in os.listdir(OUT_DIR):
    if f.endswith(".parquet"):
        print(f)

ccm_link_table_small.parquet
compustat_fundq_small.parquet
crsp_daily_stock_small.parquet
historical_volatility_small.parquet
option_prices_small.parquet
option_volume_small.parquet
security_name_history_small.parquet
security_prices_small.parquet
short_interest_small.parquet
volatility_surface_small.parquet
